In [ ]:
# @title Environment Setup & Package Installation
!pip install -q google-adk litellm "google-cloud-aiplatform[adk,agent_engines]" requests

In [ ]:
# @title Google Cloud Project & API Key Configuration
import os
import getpass

import google.auth
import vertexai

# --- Project / region ----------------------
# Colab Enterprise usually exposes the lab project automatically.
try:
    _, PROJECT_ID = google.auth.default()
except Exception:
    PROJECT_ID = None

if not PROJECT_ID:
    PROJECT_ID = input("Enter your lab Project ID: ").strip()

LOCATION = "us-central1"          # ADK / Gemini region
CLAUDE_LOCATION = "us-east5"      # Claude

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

# --- Secrets: prompted at runtime, never written to the notebook ------
MAPS_API_KEY = getpass.getpass("Google Maps Geocoding API key: ")
os.environ["MAPS_API_KEY"] = MAPS_API_KEY

# --- Models -----------
MODEL_GEMINI = "gemini-2.5-flash"

vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Project: {PROJECT_ID}")
print(f"Region:  {LOCATION}")
print(f"Maps key loaded: {bool(MAPS_API_KEY)}")

Google Maps Geocoding API key: ··········
Project: qwiklabs-gcp-04-79b727f6c308
Region:  us-central1
Maps key loaded: True


In [ ]:
# @title Tool Definition: Geocoding Function (get_lat_lon)

import os
from typing import Dict, Optional

import requests


def get_lat_lon(place: str) -> Optional[Dict[str, float]]:
    """
    Convert a place name into latitude and longitude using the Google Maps
    Geocoding API.

    Args:
        place (str): A place description, e.g. "Austin, TX" or "Chicago".

    Returns:
        Optional[Dict[str, float]]: A dictionary with "lat" and "lon" keys,
        or None if the location could not be resolved or an error occurred.
    """
    try:
        response = requests.get(
            "https://maps.googleapis.com/maps/api/geocode/json",
            params={"address": place, "key": os.environ["MAPS_API_KEY"]},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()
        if data.get("status") != "OK" or not data.get("results"):
            return None
        location = data["results"][0]["geometry"]["location"]
        return {"lat": location["lat"], "lon": location["lng"]}
    except (requests.RequestException, KeyError, ValueError):
        return None


# Smoke test
print(get_lat_lon("Denver, CO"))
print(get_lat_lon("Nowhereville, XX"))

# Expected output:
# {'lat': 39.7392358, 'lon': -104.990251}
# None

# If the first line prints None too, your key or API isn't working.
# To see the real reason, run this throwaway line in a scratch cell:
#
# requests.get("https://maps.googleapis.com/maps/api/geocode/json",
#              params={"address": "Denver, CO", "key": os.environ["MAPS_API_KEY"]}).json()

{'lat': 39.7392358, 'lon': -104.990251}
None


In [ ]:
# @title Tool Definition: Weather Forecast Function (get_extended_weather_forecast)
from typing import Dict, List, Optional
import requests

NWS_HEADERS = {"User-Agent": "(adk-skills-workshop, your-email@example.com)"}

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service
    API based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries
        containing the period name, temperature, wind, and forecast text.
        Returns None if data is unavailable or an error occurs.
    """
    try:
        # 1. Get the gridpoint for the specific lat/lon
        points_response = requests.get(
            f"https://api.weather.gov/points/{lat},{lon}",
            headers=NWS_HEADERS,
            timeout=10,
        )
        points_response.raise_for_status()

        # 2. Extract the forecast URL from the gridpoint data
        forecast_url = points_response.json()["properties"]["forecast"]

        # 3. Fetch the actual forecast
        forecast_response = requests.get(
            forecast_url,
            headers=NWS_HEADERS,
            timeout=10
        )
        forecast_response.raise_for_status()

        # 4. Parse the forecast periods
        periods = forecast_response.json()["properties"]["periods"]
        return [
            {
                "name": p["name"],
                "temperature": f"{p['temperature']} {p['temperatureUnit']}",
                "wind": f"{p['windSpeed']} {p['windDirection']}",
                "short_forecast": p["shortForecast"],
                "detailed_forecast": p["detailedForecast"]
            }
            for p in periods[:6]
        ]
    except (requests.RequestException, KeyError, ValueError):
        return None

# --- Smoke test ---
# Note: This assumes the get_lat_lon function from your previous snippet is in memory
if __name__ == "__main__":
    denver = get_lat_lon("Denver, CO")
    if denver:
        forecast = get_extended_weather_forecast(denver["lat"], denver["lon"])
        print(forecast[0] if forecast else "FAILED")
    else:
        print("Failed to geocode Denver.")

    # Non-US coordinates should return None (NWS is US-only)
    # Using Paris, France coordinates (48.8566, 2.3522) for the test
    print(get_extended_weather_forecast(48.8566, 2.3522))

{'name': 'Today', 'temperature': '97 F', 'wind': '2 to 8 mph NNW', 'short_forecast': 'Areas Of Smoke', 'detailed_forecast': 'Areas of smoke. Sunny, with a high near 97. North northwest wind 2 to 8 mph.'}
None


In [ ]:
# @title Agent Definition: Weather Assistant (Pat_Gemini)
from google.adk.agents import Agent

WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a friendly U.S. weather assistant.

Workflow:
1. When a user names a location, call get_lat_lon to resolve it to coordinates.
2. Pass those coordinates to get_extended_weather_forecast.
3. Summarize the weather in 3-5 sentences of plain language. Structure your response to include:
   - The **current weather conditions** for the immediate period (e.g., today's current temperature and conditions).
   - The **upcoming temperature range** and general outlook.
   - Anything notable about wind, precipitation, or upcoming changes.
4. If conditions are hazardous - severe storms, extreme heat or cold, high
   winds, heavy snow, ice, or flooding - begin your response with a line
   starting with "ALERT:" describing the hazard.

Rules:
- You only cover locations in the United States and its territories.
- If get_lat_lon returns None, tell the user you could not find that location
  and ask them to be more specific.
- If get_extended_weather_forecast returns None, explain that the National
  Weather Service does not cover that location, most likely because it is
  outside the United States.
- Never invent weather data. Report only what the tools return.
"""

weather_agent_gemini = Agent(
    name="Pat_Gemini",
    model=MODEL_GEMINI,
    description="Pat the Friendly Weather Agent, powered by Gemini.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

print(f"Agent '{weather_agent_gemini.name}' created on {MODEL_GEMINI}")
print(f"Tools: {[t.__name__ for t in [get_lat_lon, get_extended_weather_forecast]]}")

Agent 'Pat_Gemini' created on gemini-2.5-flash
Tools: ['get_lat_lon', 'get_extended_weather_forecast']


In [ ]:
# @title Agent Definition: Secondary Weather Agent (Pat_Gemini_Pro)
from google.adk.agents import Agent

# The ADK will route this through Vertex AI because GOOGLE_GENAI_USE_VERTEXAI="TRUE"
SECOND_GEMINI_MODEL = "gemini-1.5-pro"

weather_agent_second_gemini = Agent(
    name="Pat_Gemini_Pro",
    model=SECOND_GEMINI_MODEL,
    description="Pat the Friendly Weather Agent, powered by Gemini 1.5 Pro on Vertex AI.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

print(f"Agent '{weather_agent_second_gemini.name}' created on {SECOND_GEMINI_MODEL}")

Agent 'Pat_Gemini_Pro' created on gemini-1.5-pro


In [ ]:
# @title Test Execution: Querying the Weather Agent
from IPython.display import Markdown, display
from vertexai.preview import reasoning_engines

# 1. Create an app that hosts agent
app_flash = reasoning_engines.AdkApp(
    agent=weather_agent_gemini,
)

# 2. Create a user session
user_id = "test-user-id"
session = app_flash.create_session(user_id=user_id)

# 3. Define multiple US cities to test
test_cities = [
    "Chicago, IL",
    "Miami, FL",
    "Seattle, WA"
]

print(f"Testing Agent: {weather_agent_gemini.name}")
print("-" * 40)

# 4. Query the agent for each city
for city in test_cities:
    print(f"Querying weather for {city}...")

    last_event = None
    for event in app_flash.stream_query(
        user_id=user_id,
        # Access the dictionary key
        session_id=session["id"],
        message=f"What is the weather like in {city}?",
    ):
        last_event = event

    # Extract and display the final text response using Markdown
    if last_event and "content" in last_event and "parts" in last_event["content"]:
        response_text = last_event["content"]["parts"][0]["text"]
        display(Markdown(f"**{city}:**\n\n{response_text}"))
        print("-" * 40)
    else:
        print(f"FAILED to get a valid response for {city}.")

In [ ]:
# @title Interactive Test: Enhanced Multi-Turn Chat Loop

import string
import time
from IPython.display import Markdown, display
from vertexai.preview import reasoning_engines

# 1. Initialize the app host
app_flash = reasoning_engines.AdkApp(agent=weather_agent_gemini)

# 2. Create ONE session for the whole conversation
user_id = "interactive-user"
session = app_flash.create_session(user_id=user_id)
session_id = session["id"]

# A set of common closing statements to check against
EXIT_PHRASES = {
    "exit", "quit", "bye", "goodbye", "cya",
    "thanks", "thank you", "ok thank you", "okay thank you",
    "ok thanks", "okay thanks", "thx", "stop", "done"
}

print("==================================================")
print("  Interactive Weather Chat with Pat initialized!")
print("  Say 'thanks', 'goodbye', or 'exit' to end the chat.")
print("==================================================\n")

while True:
    # Get prompt from user
    user_prompt = input("You: ").strip()

    if not user_prompt:
        continue

    # Clean the input: lowercase it and remove all punctuation for checking
    cleaned_prompt = user_prompt.lower().translate(str.maketrans("", "", string.punctuation)).strip()

    # Check if the cleaned input matches any of our closing statements
    if cleaned_prompt in EXIT_PHRASES:
        print("\nSession ended. Stay safe and goodbye!")
        break

    print("\nPat is fetching details...")

    try:
        last_event = None
        # Passing the same session_id preserves conversation memory
        for event in app_flash.stream_query(
            user_id=user_id, session_id=session_id, message=user_prompt
        ):
            last_event = event

        # Display response
        if (
            last_event
            and "content" in last_event
            and "parts" in last_event["content"]
        ):
            response_text = last_event["content"]["parts"][0]["text"]
            display(Markdown(f"**Pat:** {response_text}"))
        else:
            print("Pat: [No response received]")

    except Exception as e:
        print(f"\nAn error occurred: {e}")

    print("\n" + "-" * 50 + "\n")

  Interactive Weather Chat with Pat initialized!
  Say 'thanks', 'goodbye', or 'exit' to end the chat.

You: hey what's the weather in Kansas city

Pat is fetching details...


**Pat:** In Kansas City today, expect patchy fog and a chance of showers and thunderstorms this morning, then partly sunny skies with a high near 84 degrees Fahrenheit. Winds will be light from the east-southeast around 5 mph. There's a 60% chance of precipitation, with minimal rainfall.

Looking ahead, temperatures will gradually rise. Tonight, it will be partly cloudy with a low around 70 degrees, and a 30% chance of showers and thunderstorms after 1 AM. Friday will be mostly sunny with a high near 89 degrees, feeling like 96 degrees with the heat index, and a slight chance of showers and thunderstorms. By Saturday, it will be sunny and hotter, with a high near 92 degrees.


--------------------------------------------------

You: bye

Session ended. Stay safe and goodbye!
